# 07 - Train a PPO Model Online

This notebook shows on-policy PPO training in Mouse Core. It mirrors the online DQN loop in `03_train_online.ipynb`, but swaps the action-value head and TD loss for an actor-critic pair:

1. Build train and held-out eval `GroupEnv`s.
2. For each of `NUM_CYCLES` cycles: collect `ROLLOUT_STEPS` env steps, run `PPO_EPOCHS` × `TRAIN_STEPS` optimizer updates, then `EVAL_STEPS` eval tasks.
3. Store each transition with its behavior `old_log_prob` in an on-policy `Datastore`.
4. Sample those rows with `DataLoader` and train with `PpoObjective` (GAE + clipped surrogate + value + entropy).
5. Clear the on-policy stores after each cycle.

The shared Mouse Core pieces stay the same: row dicts, `Datastore` / `DataLoader`, `Model.forward`, and a plain objective callable. Only the heads, action sampling, and objective change.

On-policy rollouts use **train** envs; a separate **eval** group (offset seeds) is scored greedily each cycle.


In [ ]:
from collections import deque

import torch
import numpy as np

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_group_env
from mouse_core.models.kv_policy import (
    cache_needs_rebuild,
    rebuild_starts,
    resolve_cache_bounds,
)

from mouse_core.data import DataLoader, Datastore
from mouse_core.models import Model, preferred_dtype, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionHead, SwiGLUHead
from mouse_core.objectives import PpoObjective, batch_field, sample_discrete_action


MODEL_ID = "mouse-example-model-ppo"          # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                               # number of discrete actions predicted by the policy head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
MAX_EPISODE_STEPS = 30                        # environment-specific episode limit
EPISODES_PER_TASK = 20                        # environment-specific task length
NUM_ENVS = 30                                 # number of environment streams in the GroupEnv

SEQUENCE_LENGTH = 512                         # sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step

NUM_CYCLES = 100                              # outer rollout/train/eval cycles
ROLLOUT_STEPS = 500                           # env steps per cycle (passed to run_rollout)
TRAIN_STEPS = 50                              # optimizer updates per PPO epoch (passed to run_train)
PPO_EPOCHS = 4                                # run_train calls per cycle (4 * 50 = 200 updates)
EVAL_STEPS = 1                                # tasks per eval env per cycle (passed to run_eval)

EVAL_NUM_ENVS = 4
EVAL_SEED_OFFSET = 1_000_000                  # held-out env seed stream (far from train)

LEARNING_STARTS = 15_000                      # env steps collected before the first optimizer update


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Build Environment

Online PPO uses the same `EnvConfig` / `make_group_env` setup as the other live-env notebooks. Build the **train** `GroupEnv` for on-policy rollouts and a separate **eval** `GroupEnv` (seed stream offset by `EVAL_SEED_OFFSET`) for greedy scoring between cycles.

Keep row fields consistent with the model modalities and with `PpoObjective` (`action`, `observation`, `reward`, `done`, plus rollout-only `old_log_prob`).


In [ ]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_ppo_{i}",
        reset_seed=i,
        episodes_per_task=EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},  # forwarded to the environment at task reset
        kwargs={
            "width": 8,
            "height": 8,
            "max_episode_steps": MAX_EPISODE_STEPS,
            "map_seed": i,
            "slippery_success_rate": 1.0,  # environment-specific option
            "permute_obs": True,      # environment-specific option
            "permute_actions": True,  # environment-specific option
        },
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)


In [ ]:
eval_configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"eval_frozenlake_{i}",
        reset_seed=i + EVAL_SEED_OFFSET,
        episodes_per_task=EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},
        kwargs={
            "width": 8,
            "height": 8,
            "max_episode_steps": MAX_EPISODE_STEPS,
            "map_seed": i + EVAL_SEED_OFFSET,
            "slippery_success_rate": 1.0,
            "permute_obs": True,
            "permute_actions": True,
        },
    )
    for i in range(EVAL_NUM_ENVS)
]

eval_env = make_group_env(eval_configs)


## Build The Model

PPO needs two heads on a shared backbone:

* `DiscreteActionHead` → `predictions["action"]` policy logits (also `action_head` for inference)
* `SwiGLUHead(out_features=1)` → `predictions["value"]` scalar V(s)

`old_log_prob` is **not** an encoder modality — it is stored on rollout rows and injected into `objective_data` with `batch_field` before the objective call.


In [ ]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1,
        },
    ],
)

policy_head = DiscreteActionHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)
value_head = SwiGLUHead(
    in_features=backbone.hidden_dim,
    out_features=1,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(
    encoder=encoder,
    backbone=backbone,
    heads={"action": policy_head, "value": value_head},
    action_head="action",
).train().to(device=device, dtype=preferred_dtype(device))
model.backbone.model.compile()  # optional compile step for faster repeated forwards
print(model)


## On-Policy Buffer And Contexts

Each env writes to one on-policy `Datastore` that is cleared after every training phase. Bounded `contexts` deques keep recent history for cached rollouts (in-context conditioning), separate from the on-policy buffer lifetime.


In [ ]:
stores = [Datastore(name=name) for name in env.names]
contexts = [deque(maxlen=SEQUENCE_LENGTH) for _ in env.names]


## Evaluation Phase

`run_eval` rolls out the current policy greedily on `eval_env`. It does not append to train replay. Row **contexts persist** across calls so evaluation continues mid-task; the **KV cache does not** and is rebuilt from those contexts at the start of each call. Each outer cycle passes `num_steps=EVAL_STEPS` (tasks per eval env).


In [ ]:
eval_contexts = [[] for _ in eval_env.names]

def run_eval(
    *,
    model,
    env,
    contexts: list,
    num_steps: int,
    max_cache: int = 512,
    start_cache=None,
    temperature: float = 0.0,
):
    """Greedy task-budget eval on a GroupEnv (does not append to train replay).

    ``num_steps`` is the number of tasks to complete per env stream.
    ``contexts`` persist across calls (mutated in place); the KV cache does not
    and is rebuilt from ``contexts`` at the start of each call.
    """
    model.eval()
    max_cache, start_cache = resolve_cache_bounds(max_cache, start_cache)
    n = len(env.names)
    if len(contexts) != n:
        raise ValueError(f"contexts has {len(contexts)} streams but env has {n}")
    kv_cache = None
    cached_starts = np.zeros(n, dtype=np.int64)
    cached_ends = np.zeros(n, dtype=np.int64)
    context_start = np.zeros(n, dtype=np.int64)
    eval_task_scores = [[] for _ in range(n)]
    inputs = None
    outputs = None
    env.metrics.clear()
    num_actions = env.action_space.spaces[0].n

    def _act_from_contexts(*, rebuild: bool) -> list[dict]:
        nonlocal kv_cache, cached_starts, cached_ends
        ends = np.array([len(c) for c in contexts], dtype=np.int64)
        need_rebuild = rebuild or cache_needs_rebuild(
            has_cache=kv_cache is not None,
            cached_starts=cached_starts,
            cached_ends=cached_ends,
            ends=ends,
            context_start=context_start,
            max_cache=max_cache,
            batch_complete=True,
        )
        with torch.no_grad():
            if need_rebuild:
                starts = rebuild_starts(
                    ends=ends,
                    context_start=context_start,
                    start_cache=start_cache,
                    max_cache=max_cache,
                )
                batch = [contexts[i][int(starts[i]) : int(ends[i])] for i in range(n)]
                predictions, _, kv_cache = model(batch, use_cache=True)
                cached_starts = starts
                cached_ends = ends.copy()
            else:
                batch = [
                    contexts[i][int(cached_ends[i]) : int(ends[i])] for i in range(n)
                ]
                predictions, _, kv_cache = model(batch, cache=kv_cache, use_cache=True)
                cached_ends = ends.copy()
            actions = model.get_action(
                predictions, temperature=temperature, num_actions=num_actions
            )
        random_inputs = env.sample_random_input()
        return [
            {"action": action} if contexts[i] else random_inputs[i]
            for i, action in enumerate(actions.cpu().numpy())
        ]

    while min(len(scores) for scores in eval_task_scores) < num_steps:
        if outputs is None:
            # Start of this call: rebuild KV from persisted contexts (or random if empty).
            if any(contexts):
                inputs = _act_from_contexts(rebuild=True)
            else:
                inputs = env.sample_random_input()
        else:
            task_ended = False
            for i, (inp, out) in enumerate(zip(inputs, outputs)):
                row = {**inp, **out}
                row.pop("info", None)
                contexts[i].append(row)
                if int(row["done"]) >= 3:
                    contexts[i] = []
                    context_start[i] = 0
                    task_ended = True
            inputs = _act_from_contexts(rebuild=task_ended)
        outputs = env.step(inputs)
        for i, out in enumerate(outputs):
            if int(out["done"]) >= 3:
                eval_task_scores[i].append(float(env.metrics.task_cum_rewards[i][-1]))

    scores_per_env = [scores[:num_steps] for scores in eval_task_scores]
    flat_scores = [s for scores in scores_per_env for s in scores]
    mean_task_score = (
        sum(flat_scores) / len(flat_scores) if flat_scores else float("nan")
    )
    return {
        "mean_task_score": mean_task_score,
        "task_scores": flat_scores,
        "scores_per_env": scores_per_env,
        "n_tasks": len(flat_scores),
    }


## Rollout Phase

Unlike ε-greedy DQN, PPO explores by sampling from the current categorical policy. Callers pass `num_steps=ROLLOUT_STEPS`. For each lockstep step:

1. Run `model(..., use_cache=True)` on pending context rows.
2. `sample_discrete_action` draws actions and behavior log-probs from `predictions["action"]`.
3. Step the `GroupEnv` and append flat rows that include `old_log_prob`.

`old_log_prob` sits on the same step as `action` (the transition that produced the next observation), matching the timing used by `PpoObjective`.

Row **contexts persist** across calls; the **KV cache does not** and is rebuilt from those contexts at the start of each call.


In [ ]:
def run_rollout(
    *,
    model: Model,
    env,
    stores: list[Datastore],
    contexts: list[deque],
    env_steps: int,
    num_steps: int,
) -> int:
    """Collect ``num_steps`` on-policy rows (with behavior log-probs) into datastores.

    ``contexts`` persist across calls; the KV cache does not and is rebuilt
    from ``contexts`` at the start of each call.
    """
    env.metrics.clear()
    model.eval()

    kv_cache = None
    pending_rows = [list(c) for c in contexts]

    for step in range(num_steps):
        # Need at least one context row before the policy can act.
        ready = [bool(c) for c in contexts]
        log_probs = [0.0] * len(contexts)
        if any(ready):
            with torch.no_grad():
                predictions, _, kv_cache = model(pending_rows, cache=kv_cache, use_cache=True)
            logits = predictions["action"][:, -1]
            actions_t, log_probs_t = sample_discrete_action(logits=logits, num_actions=MAX_ACTIONS)
            pending_rows = [[] for _ in contexts]
        random_inputs = env.sample_random_input()
        inputs = []
        for i in range(len(contexts)):
            if ready[i]:
                inputs.append({"action": actions_t[i].cpu().numpy()})
                log_probs[i] = float(log_probs_t[i].item())
            else:
                # First step of an empty stream: random action, dummy log-prob.
                inputs.append(random_inputs[i])
                log_probs[i] = 0.0
        outputs = env.step(inputs)
        for i, out in enumerate(outputs):
            row = {**inputs[i], **out, "old_log_prob": log_probs[i]}
            row.pop("info", None)  # keep stored rows flat
            stores[i].append(row)
            contexts[i].append(row)
            pending_rows[i].append(row)
        env_steps += len(outputs)

    if device.type == "cuda":
        torch.cuda.empty_cache()
    return env_steps


## Training Phase

After a rollout, `loader.refresh()` rescans the on-policy stores. Each PPO epoch calls `run_train(num_steps=TRAIN_STEPS)`. Before `PpoObjective`, `batch_field` copies `old_log_prob` from the sampled rows into `objective_data` (it is not an encoder modality).

Discounts match the task structure used by the DQN notebooks: undiscounted within a task, hard cut at task boundaries.


In [ ]:
loader = DataLoader(stores=stores, sequence_length=SEQUENCE_LENGTH, batch_size=BATCH_SIZE, weight_mode='per_step', num_workers=0)
objective = PpoObjective(
    gamma_step=1.0,
    gamma_episode_terminal=1.0,
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,
    gamma_task_truncated=0.0,
    gae_lambda=0.95,
    clip_eps=0.2,
    vf_coef=0.5,
    ent_coef=0.01,
    num_actions=MAX_ACTIONS,
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)

def run_train(
    *,
    model: Model,
    optimizer: torch.optim.Optimizer,
    objective: PpoObjective,
    loader: DataLoader,
    num_steps: int,
) -> tuple[torch.Tensor, dict[str, float]]:
    """Refresh the on-policy buffer and run ``num_steps`` PPO optimizer steps."""
    model.train()
    loader.refresh()

    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        batch = loader.next_batch()

        predictions, objective_data, _ = model(batch)
        objective_data["old_log_prob"] = batch_field(batch=batch, key='old_log_prob', device=objective_data.device)
        loss, metrics = objective(objective_data, predictions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    assert loss is not None
    return loss, metrics


## Run Online PPO

Each of `NUM_CYCLES` cycles calls `run_rollout(num_steps=ROLLOUT_STEPS)`, then `run_train(num_steps=TRAIN_STEPS)` for `PPO_EPOCHS` passes (once replay has `LEARNING_STARTS` rows), then clears the buffers and runs `run_eval(num_steps=EVAL_STEPS)`.


In [ ]:
env_steps = 0
grad_steps = 0
for cycle in range(NUM_CYCLES):
    env_steps = run_rollout(model=model, env=env, stores=stores, contexts=contexts, env_steps=env_steps, num_steps=ROLLOUT_STEPS)
    print(f"cycle={cycle} rollout  env_step={env_steps}")
    if env_steps >= LEARNING_STARTS:
        for _epoch in range(PPO_EPOCHS):
            loss, metrics = run_train(model=model, optimizer=optimizer, objective=objective, loader=loader, num_steps=TRAIN_STEPS)
            grad_steps += TRAIN_STEPS
        print(f"cycle={cycle} train    loss={loss.item():.4f}  policy={metrics['policy_loss']:.4f}  value={metrics['value_loss']:.4f}  grad_step={grad_steps}")
        for store in stores:
            store.clear()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    stats = run_eval(num_steps=EVAL_STEPS, max_cache=SEQUENCE_LENGTH, model=model, env=eval_env, contexts=eval_contexts)
    print(f"cycle={cycle} eval     mean_task_score={stats['mean_task_score']:.2f}/{EPISODES_PER_TASK}  n_tasks={stats['n_tasks']}")
loader.close()
env.close()
eval_env.close()
print(f'Online PPO finished ({grad_steps} optimizer steps, {env_steps} env steps).')


## Push To The Hub

Run this cell to save the PPO checkpoint. The inference notebook can load it with `load_model` / the Hub model id used here.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")
